# 📝 NB-06: BART Summarization – Thinking→Concise Answer
**Goal:** Fine-tune BART to compress Claude's long `<think>` blocks into short answers.

**Modality:** Summarization | **Model:** facebook/bart-base

In [ ]:
!pip install transformers datasets evaluate rouge_score -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
import evaluate, numpy as np, torch

tok = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

# Source: full thinking + question | Target: clean response (first 200 chars)
src = [f"{d['user']}\n\nThinking: {d['thinking'][:600]}" for d in data if d["thinking"]]
tgt = [d["response"][:200] for d in data if d["thinking"]]

def preprocess(batch):
    inp = tok(batch["src"], max_length=512, truncation=True, padding="max_length")
    with tok.as_target_tokenizer():
        lab = tok(batch["tgt"], max_length=128, truncation=True, padding="max_length")
    inp["labels"] = [[(t if t != tok.pad_token_id else -100) for t in l] for l in lab["input_ids"]]
    return inp

ds = Dataset.from_dict({"src":src,"tgt":tgt}).map(preprocess, batched=True, remove_columns=["src","tgt"]).train_test_split(0.1)
rouge = evaluate.load("rouge")

def compute_metrics(p):
    preds = tok.batch_decode(np.where(p.predictions != -100, p.predictions, tok.pad_token_id), skip_special_tokens=True)
    labels = tok.batch_decode(np.where(p.label_ids != -100, p.label_ids, tok.pad_token_id), skip_special_tokens=True)
    return rouge.compute(predictions=preds, references=labels, use_stemmer=True)

args = Seq2SeqTrainingArguments("./bart-summarizer", num_train_epochs=4,
    per_device_train_batch_size=8, predict_with_generate=True,
    fp16=torch.cuda.is_available(), evaluation_strategy="epoch", report_to="none")
Seq2SeqTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
    tokenizer=tok, data_collator=DataCollatorForSeq2Seq(tok, model), compute_metrics=compute_metrics).train()
print("✅ BART summarizer ready!")
